# US Aviation — Model Training
### GPU-Accelerated: cuML on T4

Trains two classifiers on `flights_model_ready_3class.csv` to predict flight delay severity:

| Class | Label | Definition |
|---|---|---|
| 0 | On-time / minor | DEP_DELAY ≤ 15 min |
| 1 | Moderate | 16–60 min |
| 2 | Severe | > 60 min |

Models trained:
1. **Logistic Regression** — cuML (GPU) / sklearn (CPU fallback)
2. **Random Forest** — cuML (GPU) / sklearn (CPU fallback)

Pipeline:
1. GPU check + install RAPIDS cuML
2. Load & inspect model-ready data
3. Preprocessing: encode + scale
4. Train-test split (80/20 stratified)
5. Train both models
6. Evaluate: accuracy, macro F1, weighted F1, per-class metrics
7. Confusion matrices + feature importance plots
8. Save models & training report to Google Drive

In [37]:
import subprocess
from google.colab import drive

# Detach any stale FUSE mount — does NOT delete any files on Google Drive
subprocess.run(["fusermount", "-uz", "/content/drive"], capture_output=True)

# Fresh mount
drive.mount("/content/drive", force_remount=True)
print("Drive mounted successfully.")

Mounted at /content/drive
Drive mounted successfully.


## Step 1 — GPU Check & Install

In [38]:
import subprocess, sys

def _has_gpu():
    try:
        subprocess.run(["nvidia-smi"], check=True, capture_output=True)
        return True
    except Exception:
        return False

USE_GPU = _has_gpu()

if USE_GPU:
    print("GPU detected — installing RAPIDS cuML…")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet",
         "--extra-index-url", "https://pypi.nvidia.com",
         "cuml-cu12", "cudf-cu12"],
        check=True
    )
    print("cuML installed.")
else:
    print("No GPU — falling back to sklearn (CPU).")

GPU detected — installing RAPIDS cuML…
cuML installed.


In [39]:
import json, joblib, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")

if USE_GPU:
    from cuml.linear_model import LogisticRegression as cuLR
    from cuml.ensemble import RandomForestClassifier as cuRF
    print("cuML loaded.")
else:
    from sklearn.linear_model import LogisticRegression as cuLR
    from sklearn.ensemble import RandomForestClassifier as cuRF
    print("sklearn loaded.")

cuML loaded.


## Step 2 — Configure Paths

In [40]:

MODEL_READY   = Path("/content/drive/MyDrive/aviation/flights_model_ready_3class.csv")
MODELS_DIR    = Path("/content/drive/MyDrive/aviation/models")
FIGURES_DIR   = Path("/content/drive/MyDrive/aviation/figures")
REPORT_JSON   = Path("/content/drive/MyDrive/aviation/model_training_report.json")
PER_CLASS_CSV = Path("/content/drive/MyDrive/aviation/per_class_metrics.csv")
FI_CSV        = Path("/content/drive/MyDrive/aviation/feature_importance.csv")


TARGET_COL    = "delay_class_3"
RANDOM_STATE  = 42

print("Paths configured.")
print(f"  MODEL_READY : {MODEL_READY}")
print(f"  MODELS_DIR  : {MODELS_DIR}")
print(f"  FIGURES_DIR : {FIGURES_DIR}")

Paths configured.
  MODEL_READY : /content/drive/MyDrive/aviation/flights_model_ready_3class.csv
  MODELS_DIR  : /content/drive/MyDrive/aviation/models
  FIGURES_DIR : /content/drive/MyDrive/aviation/figures


## Step 3 — Verify Drive & Load Data

In [41]:
import os

# Verify Drive is mounted and file exists before loading
drive_root = Path("/content/drive/MyDrive/aviation")

print("Drive mounted:", Path("/content/drive/MyDrive").exists())
print("Aviation folder exists:", drive_root.exists())

if drive_root.exists():
    print("\nFiles in aviation folder:")
    for f in sorted(drive_root.iterdir()):
        size_mb = f.stat().st_size / 1e6 if f.is_file() else 0
        print(f"  {f.name:<45}  {size_mb:>8.1f} MB" if f.is_file() else f"  {f.name}/")
else:
    print("\nFolder not found — re-run Cell 2 (Drive mount) and try again.")
    print("If the folder name is different, update RAW_INPUT path in Cell 7.")

Drive mounted: True
Aviation folder exists: True

Files in aviation folder:
  cleaned_flights.csv                              1005.0 MB
  cleaning_report.json                                0.0 MB
  dashboard_data_report.json                          0.0 MB
  dashboard_predictions_3class.csv                   86.4 MB
  feature_importance.csv                              0.0 MB
  figures/
  flights_model_ready_3class.csv                    351.5 MB
  flights_sample_3m.csv                             614.1 MB
  model_training_report.json                          0.0 MB
  models/
  per_class_metrics.csv                               0.0 MB
  reports/


In [42]:
# Create output directories only after Drive is confirmed mounted
assert Path("/content/drive/MyDrive").exists(), "Drive not mounted — re-run Cell 2 first."
assert MODEL_READY.exists(), f"File not found: {MODEL_READY}\nCheck the path in Cell 7."

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print("Output directories ready.")
print(f"  models/  → {MODELS_DIR}")
print(f"  figures/ → {FIGURES_DIR}")

Output directories ready.
  models/  → /content/drive/MyDrive/aviation/models
  figures/ → /content/drive/MyDrive/aviation/figures


In [43]:
print("Loading model-ready dataset…")
df = pd.read_csv(MODEL_READY, low_memory=False)
print(f"Shape: {df.shape}")
print(f"\nTarget distribution:")
dist = df[TARGET_COL].value_counts().sort_index()
labels = {0: "On-time/minor (≤15 min)", 1: "Moderate (16-60 min)", 2: "Severe (>60 min)"}
for cls, count in dist.items():
    print(f"  Class {int(cls)} — {labels[int(cls)]:28s}: {count:>8,}  ({count/len(df)*100:.1f}%)")

df.head(3)

Loading model-ready dataset…
Shape: (2912112, 23)

Target distribution:
  Class 0 — On-time/minor (≤15 min)     : 2,401,591  (82.5%)
  Class 1 — Moderate (16-60 min)        :  333,947  (11.5%)
  Class 2 — Severe (>60 min)            :  176,574  (6.1%)


,FL_DATE,AIRLINE,AIRLINE_CODE,ORIGIN,DEST,ORIGIN_STATE,DEST_STATE,CRS_DEP_TIME_MIN,CRS_ARR_TIME_MIN,CRS_ELAPSED_TIME,...,year,month,day,day_of_week,is_weekend,dep_hour,arr_hour,route,dep_time_bucket,season
0,2019-01-09,United Air Lines Inc.,UA,FLL,EWR,FL,NJ,715,901,186.0,...,2019,1,9,2,0,11,15,FLL_EWR,Morning,Winter
1,2022-11-19,Delta Air Lines Inc.,DL,MSP,SEA,MN,WA,1280,1395,235.0,...,2022,11,19,5,1,21,23,MSP_SEA,Night,Fall
2,2022-07-22,United Air Lines Inc.,UA,DEN,MSP,CO,MN,594,772,118.0,...,2022,7,22,4,0,9,12,DEN_MSP,Morning,Summer


## Step 4 — Preprocessing

Steps:
- Drop `FL_DATE` (year/month/day already extracted as numeric features)
- Drop `AIRLINE_CODE` (redundant with `AIRLINE`)
- Drop `route` (ORIGIN_DEST string — high cardinality; ORIGIN & DEST kept separately)
- OrdinalEncoder on remaining categorical columns (works with all three model types)
- StandardScaler on numeric columns (required for Logistic Regression)
- Output: single float32 numpy array X, integer array y

In [44]:
# ── Feature selection ─────────────────────────────────────────────────────────
DROP_COLS = ["FL_DATE", "AIRLINE_CODE", "route"]
df_feat   = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")

X_raw = df_feat.drop(columns=[TARGET_COL])
y     = df_feat[TARGET_COL].astype(int).values

cat_cols = X_raw.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = X_raw.select_dtypes(include=["number"]).columns.tolist()

print(f"Features      : {X_raw.shape[1]}  ({len(num_cols)} numeric, {len(cat_cols)} categorical)")
print(f"Numeric cols  : {num_cols}")
print(f"Categoric cols: {cat_cols}")

Features      : 19  (12 numeric, 7 categorical)
Numeric cols  : ['CRS_DEP_TIME_MIN', 'CRS_ARR_TIME_MIN', 'CRS_ELAPSED_TIME', 'DISTANCE', 'SCHED_BLOCK_MINS', 'year', 'month', 'day', 'day_of_week', 'is_weekend', 'dep_hour', 'arr_hour']
Categoric cols: ['AIRLINE', 'ORIGIN', 'DEST', 'ORIGIN_STATE', 'DEST_STATE', 'dep_time_bucket', 'season']


In [45]:
# ── Encode + scale ────────────────────────────────────────────────────────────
print("Encoding categorical columns…")
ord_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_cat   = ord_enc.fit_transform(X_raw[cat_cols]).astype("float32")

print("Scaling numeric columns…")
scaler  = StandardScaler()
X_num   = scaler.fit_transform(X_raw[num_cols]).astype("float32")

X = np.hstack([X_num, X_cat])   # float32, shape (n_samples, n_features)
feature_names = num_cols + cat_cols

print(f"Final feature matrix: {X.shape}  dtype={X.dtype}")
print(f"Target array        : {y.shape}  classes={np.unique(y)}")

Encoding categorical columns…
Scaling numeric columns…
Final feature matrix: (2912112, 19)  dtype=float32
Target array        : (2912112,)  classes=[0 1 2]


In [46]:
# ── Train-test split (80/20 stratified) ──────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train set: {X_train.shape[0]:,} rows")
print(f"Test  set: {X_test.shape[0]:,} rows")

Train set: 2,329,689 rows
Test  set: 582,423 rows


## Step 5 — Define Models

- **Logistic Regression** — cuML GPU solver `qn` (quasi-Newton); sklearn on CPU
- **Random Forest** — cuML GPU (float32 required, max_depth=20); sklearn on CPU.
  Class imbalance handled via `sample_weight` passed at fit time.

In [47]:
# ── Logistic Regression ───────────────────────────────────────────────────────
if USE_GPU:
    lr_model = cuLR(
        max_iter=3000,
        C=1.0,
        solver="qn",
        linesearch_max_iter=100,
    )
else:
    lr_model = cuLR(
        max_iter=3000,
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

# ── Random Forest ─────────────────────────────────────────────────────────────
if USE_GPU:
    rf_model = cuRF(
        n_estimators=100,       # reduced from 300 to fit in Colab RAM
        max_depth=16,           # cuML RF cap; kept at 16 to avoid OOM
        min_samples_split=5,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_streams=1,
    )
else:
    rf_model = cuRF(
        n_estimators=100,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=1,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

print(f"Models defined  (GPU mode: {USE_GPU})")

Models defined  (GPU mode: True)


## Step 5b — Oversample Minority Classes for Random Forest

cuML RF does not support `sample_weight` or `class_weight`.
Instead we oversample Class 1 and Class 2 up to match Class 0,
giving the model equal exposure to all three delay categories.

In [48]:
import gc


# Build balanced training set using numpy only (no DataFrame copy — saves RAM).
# 3:1:1 ratio — keeps more Class 0 examples for better overall accuracy
# while still giving the model meaningful exposure to minority classes.
CAPS = {0: 600_000, 1: 300_000, 2: 300_000}

print(f"\nOriginal class counts: { {c: int((y_train==c).sum()) for c in [0,1,2]} }")

rng = np.random.RandomState(RANDOM_STATE)
bal_X, bal_y = [], []
for cls in [0, 1, 2]:
    mask  = y_train == cls
    X_cls = X_train[mask]
    n     = len(X_cls)
    cap   = CAPS[cls]
    idx   = rng.choice(n, size=cap, replace=(n < cap))
    bal_X.append(X_cls[idx])
    bal_y.append(np.full(cap, cls, dtype="int32"))

X_train_balanced = np.vstack(bal_X).astype("float32")
y_train_balanced = np.concatenate(bal_y)

# Shuffle
shuf = rng.permutation(len(X_train_balanced))
X_train_balanced = X_train_balanced[shuf]
y_train_balanced = y_train_balanced[shuf]

del bal_X, bal_y
gc.collect()

print(f"Balanced class counts: { {c: int((y_train_balanced==c).sum()) for c in [0,1,2]} }")
print(f"Balanced train set   : {X_train_balanced.shape[0]:,} rows  (~{X_train_balanced.nbytes/1e6:.0f} MB)")


Original class counts: {0: 1921272, 1: 267158, 2: 141259}
Balanced class counts: {0: 600000, 1: 300000, 2: 300000}
Balanced train set   : 1,200,000 rows  (~91 MB)


## Step 6 — Train & Evaluate All Models

In [49]:
def to_numpy(arr):
    """Ensure predictions are a real numpy array (handles cuML output)."""
    if hasattr(arr, "to_numpy"):
        return arr.to_numpy()
    if hasattr(arr, "values"):
        return np.array(arr.values)
    return np.array(arr)


def evaluate(name, model, X_tr, y_tr, X_te, y_te, fit_kwargs=None):
    print(f"\nTraining {name}…")
    fit_kwargs = fit_kwargs or {}
    model.fit(X_tr, y_tr, **fit_kwargs)

    y_pred = to_numpy(model.predict(X_te)).astype(int)
    y_te   = to_numpy(y_te).astype(int)

    acc = accuracy_score(y_te, y_pred)
    p_mac, r_mac, f1_mac, _ = precision_recall_fscore_support(y_te, y_pred, average="macro",    zero_division=0)
    p_wt,  r_wt,  f1_wt,  _ = precision_recall_fscore_support(y_te, y_pred, average="weighted", zero_division=0)
    cm     = confusion_matrix(y_te, y_pred)
    report = classification_report(y_te, y_pred, output_dict=True, zero_division=0)

    print(f"  Accuracy        : {acc:.4f}")
    print(f"  Macro    F1     : {f1_mac:.4f}  (P={p_mac:.4f}  R={r_mac:.4f})")
    print(f"  Weighted F1     : {f1_wt:.4f}  (P={p_wt:.4f}  R={r_wt:.4f})")

    return {
        "name": name,
        "model": model,
        "accuracy": float(acc),
        "precision_macro": float(p_mac),   "recall_macro": float(r_mac),   "f1_macro": float(f1_mac),
        "precision_weighted": float(p_wt), "recall_weighted": float(r_wt), "f1_weighted": float(f1_wt),
        "confusion_matrix": cm,
        "classification_report": report,
    }


# ── Convert to cuML-friendly format if needed ─────────────────────────────────
if USE_GPU:
    import cudf
    X_train_gpu = cudf.DataFrame(X_train.astype("float32"))
    X_test_gpu  = cudf.DataFrame(X_test.astype("float32"))
    y_train_gpu = cudf.Series(y_train.astype("int32"))
    y_test_gpu  = cudf.Series(y_test.astype("int32"))
else:
    X_train_gpu, X_test_gpu = X_train, X_test
    y_train_gpu, y_test_gpu = y_train, y_test


# ── Train all models ──────────────────────────────────────────────────────────
results = []

results.append(evaluate(
    "Logistic Regression", lr_model,
    X_train_gpu, y_train_gpu, X_test_gpu, y_test_gpu
))

if USE_GPU:
    import cudf
    X_train_bal_gpu = cudf.DataFrame(X_train_balanced)
    y_train_bal_gpu = cudf.Series(y_train_balanced)
else:
    X_train_bal_gpu = X_train_balanced
    y_train_bal_gpu = y_train_balanced

results.append(evaluate(
    "Random Forest", rf_model,
    X_train_bal_gpu, y_train_bal_gpu, X_test_gpu, y_test_gpu
))

print("\nAll models trained.")


Training Logistic Regression…
  Accuracy        : 0.8247
  Macro    F1     : 0.3013  (P=0.2749  R=0.3333)
  Weighted F1     : 0.7455  (P=0.6801  R=0.8247)

Training Random Forest…
  Accuracy        : 0.7750
  Macro    F1     : 0.4289  (P=0.4466  R=0.4272)
  Weighted F1     : 0.7619  (P=0.7549  R=0.7750)

All models trained.


## Step 7 — Results Summary

In [50]:
print(f"{'Model':<22}  {'Accuracy':>8}  {'Macro F1':>9}  {'Weighted F1':>11}  {'Macro P':>8}  {'Macro R':>8}")
print("-" * 75)
for r in results:
    print(f"{r['name']:<22}  {r['accuracy']:>8.4f}  {r['f1_macro']:>9.4f}  "
          f"{r['f1_weighted']:>11.4f}  {r['precision_macro']:>8.4f}  {r['recall_macro']:>8.4f}")

# Per-class breakdown
print("\nPer-class metrics:")
class_labels = {0: "On-time/minor", 1: "Moderate", 2: "Severe"}
for r in results:
    print(f"\n  {r['name']}")
    for cls in ["0", "1", "2"]:
        c = r["classification_report"][cls]
        print(f"    Class {cls} ({class_labels[int(cls)]:13s}):  "
              f"P={c['precision']:.3f}  R={c['recall']:.3f}  F1={c['f1-score']:.3f}  support={int(c['support']):,}")

Model                   Accuracy   Macro F1  Weighted F1   Macro P   Macro R
---------------------------------------------------------------------------
Logistic Regression       0.8247     0.3013       0.7455    0.2749    0.3333
Random Forest             0.7750     0.4289       0.7619    0.4466    0.4272

Per-class metrics:

  Logistic Regression
    Class 0 (On-time/minor):  P=0.825  R=1.000  F1=0.904  support=480,319
    Class 1 (Moderate     ):  P=0.000  R=0.000  F1=0.000  support=66,789
    Class 2 (Severe       ):  P=0.000  R=0.000  F1=0.000  support=35,315

  Random Forest
    Class 0 (On-time/minor):  P=0.860  R=0.901  F1=0.880  support=480,319
    Class 1 (Moderate     ):  P=0.302  R=0.161  F1=0.210  support=66,789
    Class 2 (Severe       ):  P=0.177  R=0.219  F1=0.196  support=35,315


In [51]:
# ── Comparison bar chart ──────────────────────────────────────────────────────
comp_df = pd.DataFrame([
    {"Model": r["name"], "Accuracy": r["accuracy"],
     "Macro F1": r["f1_macro"], "Weighted F1": r["f1_weighted"]}
    for r in results
]).set_index("Model")

ax = comp_df.plot(kind="bar", figsize=(8, 5), rot=0)
ax.set_ylabel("Score")
ax.set_title("Model Comparison — 3-Class Flight Delay Classification")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "model_comparison_3class.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved → {FIGURES_DIR / 'model_comparison_3class.png'}")

Saved → /content/drive/MyDrive/aviation/figures/model_comparison_3class.png


## Step 8 — Confusion Matrices

In [52]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, r in zip(axes, results):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=r["confusion_matrix"],
        display_labels=["On-time", "Moderate", "Severe"]
    )
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(r["name"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices — 3-Class Delay Prediction", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "confusion_matrices_all.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved → {FIGURES_DIR / 'confusion_matrices_all.png'}")

Saved → /content/drive/MyDrive/aviation/figures/confusion_matrices_all.png


## Step 9 — Feature Importance

XGBoost and Random Forest both provide feature importances.
XGBoost uses `gain` (reduction in loss from each split) — the most informative measure.
cuML RF uses `impurity` (Gini decrease).

In [53]:
def plot_feature_importance(importances, feature_names, title, save_path, top_n=20):
    fi = pd.Series(importances, index=feature_names).sort_values(ascending=False)
    fi_top = fi.head(top_n).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(fi_top.index, fi_top.values)
    ax.set_title(title)
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    return fi


# ── Random Forest feature importance ─────────────────────────────────────────
rf_result = next(r for r in results if r["name"] == "Random Forest")

if USE_GPU:
    rf_importances = to_numpy(rf_result["model"].feature_importances_)
else:
    rf_importances = rf_result["model"].feature_importances_

rf_fi = plot_feature_importance(
    rf_importances,
    feature_names,
    "Random Forest Feature Importance — Top 20",
    FIGURES_DIR / "rf_feature_importance_top20.png"
)

# Save RF feature importance to CSV
fi_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_importances,
}).sort_values("importance", ascending=False)
fi_df.to_csv(FI_CSV, index=False)
print(f"Random Forest feature importance saved → {FI_CSV}")

Random Forest feature importance saved → /content/drive/MyDrive/aviation/feature_importance.csv


## Step 10 — Per-Class Metric Plots

In [54]:
rows = []
for r in results:
    for cls in ["0", "1", "2"]:
        c = r["classification_report"][cls]
        rows.append({
            "model": r["name"], "class": int(cls),
            "precision": c["precision"], "recall": c["recall"],
            "f1-score": c["f1-score"], "support": int(c["support"]),
        })
per_class_df = pd.DataFrame(rows)
per_class_df.to_csv(PER_CLASS_CSV, index=False)
print(f"Per-class metrics saved → {PER_CLASS_CSV}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
class_names = {0: "On-time", 1: "Moderate", 2: "Severe"}

for ax, metric in zip(axes, ["precision", "recall", "f1-score"]):
    pivot = per_class_df.pivot(index="class", columns="model", values=metric)
    pivot.index = [class_names[i] for i in pivot.index]
    pivot.plot(kind="bar", ax=ax, rot=0)
    ax.set_title(f"Per-Class {metric.title()}")
    ax.set_ylabel(metric.title())
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "per_class_metrics_all.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved → {FIGURES_DIR / 'per_class_metrics_all.png'}")

Per-class metrics saved → /content/drive/MyDrive/aviation/per_class_metrics.csv
Saved → /content/drive/MyDrive/aviation/figures/per_class_metrics_all.png


## Step 11 — Save Models & Training Report

In [55]:
model_paths = {}

for r in results:
    safe_name = r["name"].lower().replace(" ", "_")
    path = MODELS_DIR / f"{safe_name}_3class.joblib"
    joblib.dump(r["model"], str(path))
    model_paths[r["name"]] = str(path)
    print(f"Saved {r['name']} → {path}")

# Save OrdinalEncoder and StandardScaler (needed for inference)
joblib.dump(ord_enc, str(MODELS_DIR / "ordinal_encoder.joblib"))
joblib.dump(scaler,  str(MODELS_DIR / "standard_scaler.joblib"))
print(f"Saved preprocessors → {MODELS_DIR}")

Saved Logistic Regression → /content/drive/MyDrive/aviation/models/logistic_regression_3class.joblib
Saved Random Forest → /content/drive/MyDrive/aviation/models/random_forest_3class.joblib
Saved preprocessors → /content/drive/MyDrive/aviation/models


In [56]:
report = {
    "input_file": str(MODEL_READY),
    "dataset_shape": list(df.shape),
    "target_column": TARGET_COL,
    "feature_names": feature_names,
    "numeric_cols": num_cols,
    "categorical_cols": cat_cols,
    "dropped_cols": DROP_COLS,
    "gpu_used": USE_GPU,
    "metrics_summary": {
        r["name"]: {
            "accuracy":            r["accuracy"],
            "precision_macro":     r["precision_macro"],
            "recall_macro":        r["recall_macro"],
            "f1_macro":            r["f1_macro"],
            "precision_weighted":  r["precision_weighted"],
            "recall_weighted":     r["recall_weighted"],
            "f1_weighted":         r["f1_weighted"],
            "confusion_matrix":    r["confusion_matrix"].tolist(),
            "classification_report": r["classification_report"],
            "model_path":          model_paths[r["name"]],
        }
        for r in results
    },
}

REPORT_JSON.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(f"Training report saved → {REPORT_JSON}")

Training report saved → /content/drive/MyDrive/aviation/model_training_report.json


## Done

All outputs saved to Google Drive:

| File | Contents |
|---|---|
| `models/logistic_regression_3class.joblib` | Trained LR model |
| `models/random_forest_3class.joblib` | Trained RF model |
| `models/ordinal_encoder.joblib` | Fitted OrdinalEncoder (for inference) |
| `models/standard_scaler.joblib` | Fitted StandardScaler (for inference) |
| `figures/model_comparison_3class.png` | Accuracy / F1 bar chart |
| `figures/confusion_matrices_all.png` | Confusion matrices side-by-side |
| `figures/rf_feature_importance_top20.png` | RF feature importance |
| `figures/per_class_metrics_all.png` | Per-class P/R/F1 by model |
| `feature_importance.csv` | Random Forest importances |
| `per_class_metrics.csv` | Per-class metrics for all models |
| `model_training_report.json` | Full training report with all metrics |

### GPU vs CPU speed guide

| Model | T4 GPU time (est.) | CPU time (est.) |
|---|---|---|
| Logistic Regression | ~1–2 min | ~5–10 min |
| Random Forest (300 trees) | ~3–5 min | ~15–25 min |